PARSING FILE LOADING

In [1]:
!pip install -U bitsandbytes>=0.46.1 accelerate transformers sentencepiece

In [2]:
!pip install groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.8 MB/s eta 0:00:00


In [3]:
import bitsandbytes
print("bitsandbytes version:", bitsandbytes.__version__)


bitsandbytes version: 0.49.1


In [4]:
#CELL 1 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [5]:
#CELL 2 — Load the LaTeX File
from pathlib import Path

file_path = "/content/drive/MyDrive/FYP/Grade_4/Math/math_4_parse_file.tex"

text = Path(file_path).read_text(encoding="utf-8")



In [6]:
#CELL 2A — Load the LaTeX File
print("Total characters:", len(text))
print("\n--- First 50000 characters ---\n")
print(text[:50000])

Total characters: 250392

--- First 50000 characters ---

% This LaTeX document needs to be compiled with XeLaTeX.
\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage{ucharclasses}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[version=4]{mhchem}
\usepackage{stmaryrd}
\usepackage{soul}
\usepackage{fvextra, csquotes}
\usepackage{underscore}
\usepackage{hyperref}
\hypersetup{colorlinks=true, linkcolor=blue, filecolor=magenta, urlcolor=cyan,}
\urlstyle{same}
\usepackage{polyglossia}
\usepackage{fontspec}
\usepackage{newunicodechar}
\setmainlanguage{english}
\setotherlanguages{hindi, arabic}
\IfFontExistsTF{Noto Serif Devanagari}
{\newfontfamily\hindifont{Noto Serif Devanagari}}
{\IfFontExistsTF{Kohinoor Devanagari}
  {\newfontfamily\hindifont{Kohinoor Devanagari}}
  {\IfFontExistsTF{Devanagari MT}
    {\newfontfamily\hindifont{Devanagari MT}}
    {\IfFontExistsTF{Lohit Devanagari}
      {\newfontfamily\hindifont{Lohit Devanagari}}
      {\IfF

Normalizing parsing

In [7]:
#CELL 3 — Normalize Extra Newlines
import re

def normalize_newlines(t):
    return re.sub(r"\n{3,}", "\n\n", t)

text_norm = normalize_newlines(text)

print("After newline normalization:")
print(text_norm[:50000])


After newline normalization:
% This LaTeX document needs to be compiled with XeLaTeX.
\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage{ucharclasses}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[version=4]{mhchem}
\usepackage{stmaryrd}
\usepackage{soul}
\usepackage{fvextra, csquotes}
\usepackage{underscore}
\usepackage{hyperref}
\hypersetup{colorlinks=true, linkcolor=blue, filecolor=magenta, urlcolor=cyan,}
\urlstyle{same}
\usepackage{polyglossia}
\usepackage{fontspec}
\usepackage{newunicodechar}
\setmainlanguage{english}
\setotherlanguages{hindi, arabic}
\IfFontExistsTF{Noto Serif Devanagari}
{\newfontfamily\hindifont{Noto Serif Devanagari}}
{\IfFontExistsTF{Kohinoor Devanagari}
  {\newfontfamily\hindifont{Kohinoor Devanagari}}
  {\IfFontExistsTF{Devanagari MT}
    {\newfontfamily\hindifont{Devanagari MT}}
    {\IfFontExistsTF{Lohit Devanagari}
      {\newfontfamily\hindifont{Lohit Devanagari}}
      {\IfFontExistsTF{FreeSerif}
      

In [8]:
#CELL 4 — Remove LaTeX Comments (Optional)
def remove_comments(t):
    return re.sub(r"%.*", "", t)

text_no_comments = remove_comments(text_norm)

print("After removing comments:")
print(text_no_comments[:50000])


After removing comments:

\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage{ucharclasses}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[version=4]{mhchem}
\usepackage{stmaryrd}
\usepackage{soul}
\usepackage{fvextra, csquotes}
\usepackage{underscore}
\usepackage{hyperref}
\hypersetup{colorlinks=true, linkcolor=blue, filecolor=magenta, urlcolor=cyan,}
\urlstyle{same}
\usepackage{polyglossia}
\usepackage{fontspec}
\usepackage{newunicodechar}
\setmainlanguage{english}
\setotherlanguages{hindi, arabic}
\IfFontExistsTF{Noto Serif Devanagari}
{\newfontfamily\hindifont{Noto Serif Devanagari}}
{\IfFontExistsTF{Kohinoor Devanagari}
  {\newfontfamily\hindifont{Kohinoor Devanagari}}
  {\IfFontExistsTF{Devanagari MT}
    {\newfontfamily\hindifont{Devanagari MT}}
    {\IfFontExistsTF{Lohit Devanagari}
      {\newfontfamily\hindifont{Lohit Devanagari}}
      {\IfFontExistsTF{FreeSerif}
        {\newfontfamily\hindifont{FreeSerif}}
        {\newfontfam

In [9]:
def normalize_latex_math(text):
    # ---- FRACTIONS ----
    text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'(\1/\2)', text)

    # ---- OPERATORS spacing ----
    text = re.sub(r'\s*([+\-=])\s*', r' \1 ', text)

    # ---- MULTIPLICATION FIX ----
    # Replace only real multiplication, NOT LaTeX commands
    text = text.replace('\\times', '×')

    # Replace * only when between numbers (5*3 → 5×3)
    text = re.sub(r'(\d)\s*\*\s*(\d)', r'\1×\2', text)

    # ---- REMOVE \text{} ----
    text = re.sub(r'\\text\{([^}]+)\}', r'\1', text)

    # ---- REMOVE excessive whitespace ----
    text = re.sub(r'\s+', ' ', text).strip()

    return text


In [10]:
#CELL 5 — Final Whitespace Trim
final_text = text_no_comments.strip()

print("Final cleaned text preview:")
print(final_text[:50000])


Final cleaned text preview:
\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage{ucharclasses}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[version=4]{mhchem}
\usepackage{stmaryrd}
\usepackage{soul}
\usepackage{fvextra, csquotes}
\usepackage{underscore}
\usepackage{hyperref}
\hypersetup{colorlinks=true, linkcolor=blue, filecolor=magenta, urlcolor=cyan,}
\urlstyle{same}
\usepackage{polyglossia}
\usepackage{fontspec}
\usepackage{newunicodechar}
\setmainlanguage{english}
\setotherlanguages{hindi, arabic}
\IfFontExistsTF{Noto Serif Devanagari}
{\newfontfamily\hindifont{Noto Serif Devanagari}}
{\IfFontExistsTF{Kohinoor Devanagari}
  {\newfontfamily\hindifont{Kohinoor Devanagari}}
  {\IfFontExistsTF{Devanagari MT}
    {\newfontfamily\hindifont{Devanagari MT}}
    {\IfFontExistsTF{Lohit Devanagari}
      {\newfontfamily\hindifont{Lohit Devanagari}}
      {\IfFontExistsTF{FreeSerif}
        {\newfontfamily\hindifont{FreeSerif}}
        {\newfontf

In [11]:
#CELL 5B — Remove Page Footer / Header Noise
def remove_page_noise(text):

    patterns = [
        r'Not For Sale\s*-\s*PESRP',
        r'Mathematics\s*4\s*\|\s*Unit\s*\d+\s*:\s*[A-Za-z ]+',
        r'Mathematics\s*4\s*Unit\s*\d+\s*:\s*[A-Za-z ]+',
        r'\n\d{1,3}\\\\',      # page numbers like 36\\
        r'\n\d{1,3}\n',        # standalone page numbers
    ]

    for p in patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE)

    return text


final_text2 = remove_page_noise(text)


In [12]:
#CELL 6 — Save Clean File (Optional)
clean_path = "/content/drive/MyDrive/FYP/Grade_4/Math/final_math_parse.tex"
Path(clean_path).write_text(final_text2, encoding="utf-8")

print("Saved cleaned file to:", clean_path)


Saved cleaned file to: /content/drive/MyDrive/FYP/Grade_4/Math/final_math_parse.tex


Chunking

In [13]:
#CELL 7 — Load Clean LaTeX File
file_path = "/content/drive/MyDrive/FYP/Grade_4/Math/final_math_parse.tex"

text = Path(file_path).read_text(encoding="utf-8")

print("Characters:", len(text))
print("\n--- Preview ---\n")
print(text[:5000])


Characters: 240280

--- Preview ---

% This LaTeX document needs to be compiled with XeLaTeX.
\documentclass[11pt]{article}
\usepackage[utf8]{inputenc}
\usepackage{ucharclasses}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage[version=4]{mhchem}
\usepackage{stmaryrd}
\usepackage{soul}
\usepackage{fvextra, csquotes}
\usepackage{underscore}
\usepackage{hyperref}
\hypersetup{colorlinks=true, linkcolor=blue, filecolor=magenta, urlcolor=cyan,}
\urlstyle{same}
\usepackage{polyglossia}
\usepackage{fontspec}
\usepackage{newunicodechar}
\setmainlanguage{english}
\setotherlanguages{hindi, arabic}
\IfFontExistsTF{Noto Serif Devanagari}
{\newfontfamily\hindifont{Noto Serif Devanagari}}
{\IfFontExistsTF{Kohinoor Devanagari}
  {\newfontfamily\hindifont{Kohinoor Devanagari}}
  {\IfFontExistsTF{Devanagari MT}
    {\newfontfamily\hindifont{Devanagari MT}}
    {\IfFontExistsTF{Lohit Devanagari}
      {\newfontfamily\hindifont{Lohit Devanagari}}
      {\IfFontExistsTF{FreeSerif

In [14]:
#CELL 8 — Define Regex Patterns
import re

SECTION_RE = re.compile(r'\\section\*?\{(.+?)\}')
SUBSECTION_RE = re.compile(r'\\subsection\*?\{(.+?)\}')

print("Regex updated for starred and unstarred sections")


Regex updated for starred and unstarred sections


In [15]:
#CELL 9A — Parse LaTeX into Nodes
def parse_latex_structure(text):
    tokens = []

    for m in SECTION_RE.finditer(text):
        tokens.append(("section", m.start(), m.group(1)))

    for m in SUBSECTION_RE.finditer(text):
        tokens.append(("subsection", m.start(), m.group(1)))

    tokens.sort(key=lambda x: x[1])

    nodes = []
    for i,(level,pos,title) in enumerate(tokens):
        end = tokens[i+1][1] if i+1 < len(tokens) else len(text)
        nodes.append({
            "level": level,
            "title": title.strip(),
            "raw": text[pos:end]
        })
    return nodes

nodes = parse_latex_structure(text)
print("Nodes:", len(nodes))


Nodes: 365


In [16]:
#CELL 9B — Parse LaTeX into Nodes(Inspection)
print("Sections:", text.count("\\section"))
print("Subsections:", text.count("\\subsection"))


Sections: 181
Subsections: 184


In [17]:
#CELL 9C — Parse LaTeX into Nodes(Inspection)

for n in nodes[:100]:
    print(n["level"], "->", n["title"])


section -> Mathematics
section -> Grade 4
section -> Contents
subsection -> National Review Committee Members
section -> Learning Outcomes
section -> Numbers up to One Hundred Thousand
subsection -> Try Yourself
subsection -> Try Yourself
subsection -> Try Yourself
subsection -> Try It! Challenge
section -> Exercise 1
section -> Comparing and Ordering Numbers
subsection -> Try Yourself
subsection -> Key Fact
subsection -> Try Yourself
subsection -> Try Yourself
subsection -> Try It! Challenge
section -> Exercise 2
section -> I have learnt to:
subsection -> Vocabulary
section -> Review Exercise
subsection -> 3. Write the following numbers in the expanded form:
subsection -> 4. Write the following in numerals:
subsection -> 5. Write the place and place value of the coloured digits.
section -> Addition and Subtraction
section -> Learning Outcomes
section -> Addition
subsection -> Try Yourself
subsection -> Try Yourself
subsection -> Challenge
subsection -> Try It!
section -> Exercise 1
se

In [18]:
#CELL 10A — Build Tree (Attach Subsections to Sections)
def build_tree(nodes):
    tree=[]
    current=None

    for n in nodes:
        if n["level"]=="section":
            current=n
            current["children"]=[]
            tree.append(current)
        else:
            current["children"].append(n)
    return tree

tree = build_tree(nodes)


In [19]:
#CELL 10B — Build Tree (Attach Subsections to Sections)(Inspection)

tree = build_tree(nodes)

print("Total top-level sections:", len(tree))
print("20TH section:", tree[20]["title"])
print("Children in 20TH section:", len(tree[20]["children"]))


Total top-level sections: 181
20TH section: Multiplication
Children in 20TH section: 1


In [20]:
#CELL 10C — Build Tree (Attach Subsections to Sections)(Inspection)

sample = tree[100]

print("Section:", sample["title"])
print("Children:")

for c in sample["children"][:5]:
    print(" -", c["title"])


Section: Exercise 2
Children:


In [21]:
# 11 — Pedagogy Classifier
def classify_pedagogy(title):
    t=title.lower()
    if "try yourself" in t: return "practice"
    if "try it" in t or "challenge" in t: return "challenge"
    if "key fact" in t: return "rule"
    if "exercise" in t: return "assessment"
    if "vocabulary" in t: return "glossary"
    if "i have learnt" in t: return "learning_outcomes"
    return "explanation"



print(classify_pedagogy("Try Yourself"))
print(classify_pedagogy("Exercise 2"))
print(classify_pedagogy("Key Fact"))


practice
assessment
rule


In [22]:
#CELL 12 — Question Splitter
QUESTION_RE = re.compile(r'\n\s*(\d+\.|\([a-z]\))')

def split_questions(text):
    parts = QUESTION_RE.split(text)
    questions = []

    for i in range(1, len(parts), 2):
        qid = parts[i].strip()
        body = parts[i+1].strip()

        questions.append({
            "question_id": qid,
            "text": body
        })

    return questions


sample = """
1. What is 5+5?
2. Find value of 9×2.
(a) Easy
(b) Hard
"""

print(split_questions(sample))


[{'question_id': '1.', 'text': 'What is 5+5?'}, {'question_id': '2.', 'text': 'Find value of 9×2.'}, {'question_id': '(a)', 'text': 'Easy'}, {'question_id': '(b)', 'text': 'Hard'}]


In [23]:
#CELL 13A — Build Final Chunks
def build_chunks(tree):
    chunks = []

    for sec in tree:
        unit = sec["title"]
        sec_type = classify_pedagogy(unit)

        chunks.append({
            "unit": unit,
            "title": unit,
            "pedagogy": sec_type,
            "content": sec["raw"]
        })

        for sub in sec.get("children", []):
            pedagogy = classify_pedagogy(sub["title"])

            if sec_type == "assessment":
                qs = split_questions(sub["raw"])
                for q in qs:
                    chunks.append({
                        "unit": unit,
                        "title": sub["title"],
                        "pedagogy": "question",
                        "question_id": q["question_id"],
                        "content": q["text"]
                    })
            else:
                chunks.append({
                    "unit": unit,
                    "title": sub["title"],
                    "pedagogy": pedagogy,
                    "content": sub["raw"]
                })

    return chunks


chunks = build_chunks(tree)

print("Total chunks:", len(chunks))
print("\nchunk:\n", chunks[20])

print("\nchunk:\n", chunks[78])
print("\nchunk:\n", chunks[138])
print("\nchunk:\n", chunks[400])
print("\nchunk:\n", chunks[276])


Total chunks: 453

chunk:
 {'unit': 'Review Exercise', 'title': 'Review Exercise', 'pedagogy': 'assessment', 'content': "\\section*{Review Exercise}\n\\begin{enumerate}\n  \\item Choose the correct options and fill in the blanks.\\\\\n(a) The smallest 6-digit number is \\_\\_\\_\\_\\_\\_\\_\\_\\_\\_.\\\\\n(i) 111 111 (ii) 100 000 (iii) 101 010 (iv) 111 000\n\n(b) Comparison of numbers always starts from the \\_\\_\\_\\_\\_\\_\\_\\_\\_\\_.\\\\\n(i) right (ii) left (iii) last (iv) above\n\n(c) In number 38 101, the place value of digit '8' is \\_\\_\\_\\_\\_\\_\\_\\_\\_\\_.\\\\\n(i) 800 (ii) 8 (iii) 80 (iv) 8000\n\n(d) The greatest 5-digit number is \\_\\_\\_\\_\\_\\_\\_\\_\\_\\_.\\\\\n(i) 91 100 (ii) 90 101 (iii) 99 999 (iv) 90 000\n\n(e) 34 011 is greater than \\_\\_\\_\\_\\_\\_\\_\\_\\_\\_.\\\\\n(i) 34 010 (ii) 34 111 (iii) 34 210 (iv) 34 212\n\n(f) 31 108 is smaller than \\_\\_\\_\\_\\_\\_\\_\\_\\_\\_.\\\\\n(i) 31 106 (ii) 31 107 (iii) 30 100 (iv) 31 109\n\n  \\item Write the followi

In [24]:
#CELL 13b — Build Final Chunks(Inspection)

lengths = [len(c["content"]) for c in chunks]
print("Min:", min(lengths))
print("Max:", max(lengths))
print("Avg:", sum(lengths)//len(lengths))


Min: 5
Max: 3449
Avg: 517


In [25]:
#CELL 13C — Build Final Chunks(Inspection)
import random

for i in range(50):
    c = random.choice(chunks)
    print("\n--------------------")
    print("Unit:", c["unit"])
    print("Title:", c["title"])
    print("Pedagogy:", c["pedagogy"])
    print("Content preview:\n", c["content"][:300])



--------------------
Unit: Learning Outcomes
Title: Learning Outcomes
Pedagogy: explanation
Content preview:
 \section*{Learning Outcomes}
\textbf{After completing this unit, you will be able to:}

\begin{itemize}
  \item Recognize a decimal number as an alternative way of writing a fraction.
  \item Express a decimal number as a fraction whose denominator is 10, 100 or 1000.
  \item Identify and recognize 

--------------------
Unit: Unit 6 Geometry
Title: Exercise 2
Pedagogy: assessment
Content preview:
 \subsection*{Exercise 2}
\begin{enumerate}
  \item (a) acute angle (b) right angle (c) obtuse angle (d) acute angle
\end{enumerate}



--------------------
Unit: Addition
Title: Challenge
Pedagogy: challenge
Content preview:
 \subsection*{Challenge}



--------------------
Unit: Time
Title: Exercise 1
Pedagogy: assessment
Content preview:
 \subsection*{Exercise 1}
\begin{enumerate}
  \item (a) 6:00 a.m. (b) 7:45 a.m. (c) 12:30 p.m. (d) 4:30 p.m. (e) 8:00 p.m. (f) 10:25 p.m.
  \item 

In [26]:
#CELL 14 — Save as JSON chunks
import json

output_path = "/content/drive/MyDrive/FYP/Grade_4/Math/math4_chunks.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print("Saved to:", output_path)


Saved to: /content/drive/MyDrive/FYP/Grade_4/Math/math4_chunks.json


In [27]:
#CELL 15 — Install Libraries
!pip install chromadb sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fo

In [28]:
#CELL 16 — Load Embedder
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [29]:
# CELL 17 — Connect Shared ChromaDB
import chromadb
from chromadb.config import Settings

SHARED_DB="/content/drive/MyDrive/FYP/OneSharedChromaDB"

client = chromadb.PersistentClient(
    path=SHARED_DB,
    settings=Settings(anonymized_telemetry=False)
)

collection = client.get_or_create_collection(
    name="ptb_textbooks",
    metadata={"hnsw:space":"cosine"}
)

print("Connected to ChromaDB")
print("Total vectors:", collection.count())


Connected to ChromaDB
Total vectors: 1848


In [30]:
# CELL 18 — Delete Old MATH4
print("Before delete:", collection.count())

collection.delete(where={"doc_id":"MATH4"})

print("After delete:", collection.count())
print("✅ Old MATH4 vectors removed")


Before delete: 1848
After delete: 1848
✅ Old MATH4 vectors removed


In [31]:
# CELL 19 — Prepare Metadata
DOC_ID="MATH4"
GRADE=4
SUBJECT="mathematics"
BOOK="MATH4"

import hashlib

def hash_text(t):
    return hashlib.md5(t.encode()).hexdigest()

ids=[]
texts=[]
metas=[]

for i,c in enumerate(chunks):
    cid=f"{DOC_ID}_chunk_{i:06d}"

    ids.append(cid)
    texts.append(c["content"])

    metas.append({
        "chunk_id":cid,
        "doc_id":DOC_ID,
        "grade":GRADE,
        "subject":SUBJECT,
        "book":BOOK,
        "unit":c.get("unit",""),
        "chapter":c.get("unit",""),
        "section":c.get("title",""),
        "block_type":c.get("pedagogy","body"),
        "difficulty":"medium",
        "source_type":"latex",
        "source_pdf":"",
        "text_hash":hash_text(c["content"]),
        "embed_model":"sentence-transformers/all-MiniLM-L6-v2"
    })

print("Prepared:", len(ids), "math chunks")


Prepared: 453 math chunks


In [32]:
# CELL 20 — Embed + Cache
import numpy as np, os
from tqdm import tqdm

MATH_DIR="/content/drive/MyDrive/FYP/Grade_4/Math"
cache_dir=os.path.join(MATH_DIR,"embeddings_cache")
os.makedirs(cache_dir,exist_ok=True)

CACHE_FILE=os.path.join(cache_dir,"MATH4_embeddings.npz")

BATCH=64

if os.path.exists(CACHE_FILE):
    print("✅ Loading cached embeddings")
    data=np.load(CACHE_FILE,allow_pickle=True)
    embeddings=data["embeddings"]
else:
    print("⚙️ Computing embeddings...")
    vecs=[]
    for i in tqdm(range(0,len(texts),BATCH)):
        vecs.append(
            embedder.encode(
                texts[i:i+BATCH],
                normalize_embeddings=True
            )
        )

    embeddings=np.vstack(vecs).astype("float32")

    np.savez_compressed(
        CACHE_FILE,
        embeddings=embeddings
    )

    print("✅ Embeddings cached")

print("Embeddings shape:", embeddings.shape)


⚙️ Computing embeddings...


100%|██████████| 8/8 [00:01<00:00,  5.08it/s]

✅ Embeddings cached
Embeddings shape: (453, 384)


In [33]:
# CELL 21 — Upsert into Chroma
from tqdm import tqdm

for i in tqdm(range(0,len(ids),200)):
    j=i+200
    collection.upsert(
        ids=ids[i:j],
        embeddings=embeddings[i:j].tolist(),
        documents=texts[i:j],
        metadatas=metas[i:j]
    )

print("Total vectors now:", collection.count())
print("✅ Fresh MATH4 stored")


100%|██████████| 3/3 [00:02<00:00,  1.35it/s]

Total vectors now: 2301
✅ Fresh MATH4 stored


In [34]:
# CELL 22 — Verify
check = collection.get(where={"doc_id":"MATH4"}, limit=5)
print("MATH4 vectors found:", len(check["ids"]))


MATH4 vectors found: 5


In [35]:
#CELL 21B Inspection
print("Total vectors in shared DB:", collection.count())


Total vectors in shared DB: 2301


In [36]:
#CELL 21C Inspection
res = collection.get(where={"doc_id":"MATH4"}, limit=5)
print("Math4 samples:", len(res["ids"]))
print(res["metadatas"][0])


Math4 samples: 5
{'subject': 'mathematics', 'difficulty': 'medium', 'source_type': 'latex', 'book': 'MATH4', 'doc_id': 'MATH4', 'source_pdf': '', 'block_type': 'explanation', 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2', 'chunk_id': 'MATH4_chunk_000000', 'chapter': 'Mathematics', 'grade': 4, 'unit': 'Mathematics', 'text_hash': '54d852735dfe8a208741e3ea2cdaeb63', 'section': 'Mathematics'}


In [37]:
#CELL 21D Inspection
res = collection.get(where={"doc_id":"GS4"}, limit=5)
print("Science samples:", len(res["ids"]))
print(res["metadatas"][0])


Science samples: 5
{'grade': 4, 'chapter': 'General Science', 'source_pdf': 'GS 4.pdf', 'source_type': 'markdown_clean', 'subject': 'science', 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2', 'book': 'GS4', 'chunk_id': 'GS4_chunk_000000', 'section': 'Grade 4', 'doc_id': 'GS4', 'unit': '', 'text_hash': 'adb0eecdfbf969f442afc37568974c11', 'block_type': 'BODY'}


In [38]:
#CELL 22 Inspection
q = "How do we add fractions?"
emb = embedder.encode([q], normalize_embeddings=True)

res = collection.query(
    query_embeddings=emb.tolist(),
    n_results=5
)

for d in res["documents"][0]:
    print(d[:500], "\n")


\section*{Addition and Subtraction of Fractions}

 

\section*{Mixed numbers}
\begin{displayquote}
Subhan has two packs of juice. He drinks one full and other half pack. How can we write it in fraction?
\end{displayquote}

We can show it with the help of figure as:

$$1\frac{1}{2} = 1 + \frac{1}{2}$$

Mixed number is the sum of whole number '1' and proper fraction '$\frac{1}{2}$'.

 

\subsection*{Try It! Challenge}
If the sum of two decimals is 12.8, then find the first decimal fraction if the second decimal fraction is 7.9. What do we add in 12.8 to get 25.8?

 

\section*{Conversion of fractions}

 

\subsection*{Try It! Challenge}
Maryam wants to make three kinds of biscuits. She needs $\frac{2}{7}$ cup of flour for the first kind, $\frac{5}{7}$ cup of flour for the second and $\frac{3}{7}$ cup of flour for the third. How much flour will she need to make three kinds of biscuits?

Make groups of the students, ask them to write different fractions (with same denominators) in notebook

In [39]:
#CELL 22C Verify

print(collection.count())


2301


In [40]:
#CELL 23 — Query with Metadata Filtering

def make_where(filters: dict):
    parts = [{k:v} for k,v in filters.items()]
    if len(parts) == 1:
        return parts[0]
    return {"$and": parts}


def retrieve(query, grade=None, subject=None, top_k=5):

    q_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    )

    filters = {}
    if grade is not None:
        filters["grade"] = grade
    if subject is not None:
        filters["subject"] = subject

    where = make_where(filters) if filters else None

    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        where=where
    )

    return results["documents"][0], results["metadatas"][0]


docs, metas = retrieve(
    "Write the place value of each digit in 200.149 and express in expanded form.",
    grade=4,
    subject="mathematics"
)

print(docs[0][:400])
print(metas[0])


\subsection*{Try It!}
Write the place value of each digit in 200.149 and express in expanded form.
\end{displayquote}


{'block_type': 'challenge', 'source_pdf': '', 'section': 'Try It!', 'embed_model': 'sentence-transformers/all-MiniLM-L6-v2', 'doc_id': 'MATH4', 'grade': 4, 'text_hash': '777f6876046b105e557bc3ba21444712', 'chapter': 'Place Value of Digits in Decimals', 'book': 'MATH4', 'source_type': 'latex', 'unit': 'Place Value of Digits in Decimals', 'chunk_id': 'MATH4_chunk_000141', 'subject': 'mathematics', 'difficulty': 'medium'}


In [41]:
docs,_ = retrieve("decimal fraction", subject="mathematics")
print(len(docs))


5


In [42]:
docs,_ = retrieve("states of matter", subject="science")
print(len(docs))


5


In [43]:
# 24 Load All Math Docs from Shared Chroma (for BM25)
all_data = collection.get(
    where={"doc_id":"MATH4"},
    include=["documents","metadatas"]
)

docs = all_data["documents"]
metas = all_data["metadatas"]
ids   = all_data["ids"]

print("Total MATH4 docs loaded:", len(docs))
print(docs[0][:200])


Total MATH4 docs loaded: 453
\section*{Mathematics}




In [44]:
print(len(ids), len(docs), len(metas))


453 453 453


In [45]:
#CELL 25 — Tokenizer (same as science)
import re

def tokenize(s):
    s = (s or "").lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    return s.split()


In [46]:
#CELL 26A — Install Dependencies
!pip install rank_bm25 numpy

In [47]:
#CELL 26B — Build BM25 Index
from rank_bm25 import BM25Okapi

bm25_tokens = [tokenize(t) for t in docs]
bm25 = BM25Okapi(bm25_tokens)

print("BM25 built on math corpus")


BM25 built on math corpus


In [48]:
#CELL 26C — Query Rewriter (Before Retrieval)

def rewrite_query(q):

    q = q.lower().strip()

    # normalize informal words
    q = q.replace("plz", "please")
    q = q.replace(" n ", " and ")
    q = q.replace("?", "")

    # collapse spaces
    q = re.sub(r"\s+", " ", q)

    return q


In [49]:
#CELL 27 — STEP B2: Intent Query Router
def detect_intent_math(q):

    q = q.lower()

    if any(x in q for x in ["solve","find","calculate","work out"]):
        return "PROBLEM"

    if any(x in q for x in ["example","show"]):
        return "EXAMPLE"

    if any(x in q for x in ["define","what is","meaning of"]):
        return "DEFINITION"

    return "EXPLAIN"


In [50]:
#CELL 28 — Allowed Block Types (Math)
def allowed_block_types_math(intent):

    if intent == "PROBLEM":
        return {"question","practice","explanation"}

    if intent == "DEFINITION":
        return {"glossary","rule","explanation"}

    return {"explanation","rule","practice"}


In [51]:
# CELL 29 — Dense Retrieve (Math Only)
import numpy as np

def dense_retrieve_math(query, k=40):

    qv = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].tolist()

    out = collection.query(
        query_embeddings=[qv],
        n_results=k,
        where={"doc_id":"MATH4"},
        include=["documents","metadatas","distances"]
    )

    res = []

    for i in range(len(out["documents"][0])):
        sim = 1.0 - float(out["distances"][0][i])
        res.append({
            "id": out["ids"][0][i],
            "text": out["documents"][0][i],
            "meta": out["metadatas"][0][i],
            "dense_sim": sim
        })

    return res


In [52]:
#CELL 30 — BM25 Retrieve
def bm25_retrieve_math(query, k=40):

    toks = tokenize(query)
    scores = bm25.get_scores(toks)

    top_idx = np.argsort(scores)[::-1][:k]

    res = []
    for i in top_idx:
        res.append({
            "id": ids[i],
            "text": docs[i],
            "meta": metas[i],
            "bm25": float(scores[i])
        })

    return res


In [53]:
#CELL 31 — Hybrid Merge
def hybrid_candidates_math(query):

    dense = dense_retrieve_math(query)
    sparse = bm25_retrieve_math(query)

    merged = {}

    for r in dense:
        merged[r["id"]] = {
            **r,
            "bm25": 0.0
        }

    for r in sparse:
        if r["id"] not in merged:
            merged[r["id"]] = {
                **r,
                "dense_sim": 0.0
            }
        else:
            merged[r["id"]]["bm25"] = r["bm25"]

    items = list(merged.values())

    bm = np.array([x["bm25"] for x in items])
    bm_norm = (bm - bm.min()) / (bm.max() - bm.min() + 1e-9)

    for i,it in enumerate(items):
        it["bm25_norm"] = float(bm_norm[i])
        it["hybrid_score"] = 0.65*it["dense_sim"] + 0.35*it["bm25_norm"]

    items.sort(key=lambda x: x["hybrid_score"], reverse=True)

    return items


In [54]:
#CELL 32 Chapter Hint
from collections import Counter

def pick_chapter_hint(cands):

    chapters = [c["meta"].get("chapter","") for c in cands]
    chapters = [c for c in chapters if c]

    if not chapters:
        return None, 0.0

    cnt = Counter(chapters)
    top_ch, top_n = cnt.most_common(1)[0]
    conf = top_n / len(chapters)

    return top_ch, conf


In [55]:
# CELL 33 — Reranker
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranker loaded


In [56]:
# CELL 34 — Neural Rerank + Intent Filtering

def rerank_math(query, candidates, intent, top_k=6):

    if len(candidates) == 0:
        return []

    # -------------------------
    # 1) Intent filtering
    # -------------------------
    allow = allowed_block_types_math(intent)

    filt = []
    for c in candidates:
        bt = c["meta"].get("block_type","")
        if bt in allow:
            filt.append(c)

    if len(filt) == 0:
        return []

    # -------------------------
    # 2) Pure neural reranking
    # -------------------------
    TOPN = min(30, len(filt))
    subset = filt[:TOPN]

    pairs = [(query, c["text"]) for c in subset]
    scores = reranker.predict(pairs)

    for c, s in zip(subset, scores):
        c["rerank_score"] = float(s)

    # Remaining get tiny score
    for c in filt[TOPN:]:
        c["rerank_score"] = 0.0

    filt.sort(key=lambda x: x["rerank_score"], reverse=True)

    return filt[:top_k]


In [57]:
# CELL 35 — Final Retrieve (Math)
def retrieve_math(query, k=6):

    original_query = query
    rewritten_query = rewrite_query(query)

    intent = detect_intent_math(rewritten_query)

    cands = hybrid_candidates_math(rewritten_query)

    ch_hint, ch_conf = pick_chapter_hint(cands[:10])

    final = rerank_math(rewritten_query, cands, intent, top_k=k)

    return {
        "original_query": original_query,
        "rewritten_query": rewritten_query,
        "intent": intent,
        "chapter_hint": ch_hint,
        "chapter_conf": round(ch_conf,3),
        "hits": final
    }


In [58]:
#CELL 36-1 Query testing
out = retrieve_math(
    "Write the place value of each digit in 200.149 and express in expanded form."
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])



Original Query : Write the place value of each digit in 200.149 and express in expanded form.
Rewritten Query: write the place value of each digit in 200.149 and express in expanded form.
Intent: EXPLAIN
Chapter hint: Numbers up to One Hundred Thousand | conf: 0.3

RANK 1
block: practice | chapter: Numbers up to One Hundred Thousand
\subsection*{Try Yourself}
Tell the place and place value of every digit in 42 918. Also write this number in words.

'6' is at the ten thousands place and its place value $= 6 \times 10\ 000 = 60\ 000$\\
'9' is at the thousands place and its place value $= 9 \times 1\ 000 = 9\ 000$\\
'2' is at the hundreds place and its place value $= 2 \times 100 = 200$\\
'7' is at the tens place and its place value $= 7 \times 10 = 70$\\
'3' is at the ones place and its place value $= 3 \times 1 = 3$

\begin{

RANK 2
block: practice | chapter: Multiplication
\subsection*{Try Yourself}
Multiply the greatest 4-digit number with the greatest 3-digit number. Multiply the sma

In [59]:
#CELL 36-2 Query testing

out = retrieve_math(
     "What is expanded form?"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])


Original Query : What is expanded form?
Rewritten Query: what is expanded form
Intent: DEFINITION
Chapter hint: Place Value of Digits in Decimals | conf: 0.2

RANK 1
block: explanation | chapter: Glossary
\section*{Glossary}
\textbf{Expanded form:} To write the number as sum of place values is called expanded form.\\
\textbf{Ascending order:} The arrangement of numbers from the smallest to the greatest is called an ascending order.\\
\textbf{Descending order:} The arrangement of number from the greatest to the smallest is called descending order.\\
\textbf{Number pattern:} The number pattern is the sequence of numbers that can increase, decrease and repeat according to some rules.\\
\textbf{Prime

RANK 2
block: explanation | chapter: Improper fractions
\section*{Improper fractions}
A girl says: "Consider the following figures:"

Two diagrams are shown:

\begin{enumerate}
  \item A square divided into three equal vertical rectangles. All three rectangles are shaded pink.
$$\frac{3}{3}$$

In [60]:
#CELL 36-3 Query testing

out = retrieve_math(
    "What is a proper fraction?"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])



Original Query : What is a proper fraction?
Rewritten Query: what is a proper fraction
Intent: DEFINITION
Chapter hint: Mixed numbers | conf: 0.2

RANK 1
block: explanation | chapter: Proper fractions
\section*{Proper fractions}
A boy says: "I have a chocolate. I ate its 3 pieces out of 10 equal parts. It means that I have eaten $\frac{3}{10}$ of the chocolate."

An image shows a chocolate bar with some pieces broken off.

It can be shown with the help of a diagram.

A rectangle divided into 10 equal vertical strips. The last 3 strips on the right are shaded teal, representing $\frac{3}{10}$.

\begin{displayquote}
\textbf{Key Fact}

Fraction with numerator smaller than its denominator is call

RANK 2
block: rule | chapter: Mixed numbers
\subsection*{Key Fact}
A mixed number / mixed fraction consists of a whole number and a proper fraction.



RANK 3
block: rule | chapter: Improper fractions
\subsection*{Key Fact}
The fraction with numerator greater than its denominator is called improp

In [61]:
#CELL 36-4 Query testing
out = retrieve_math(
    "Find LCM of 6 and 8"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])


Original Query : Find LCM of 6 and 8
Rewritten Query: find lcm of 6 and 8
Intent: PROBLEM
Chapter hint: Common Multiples | conf: 0.2

RANK 1
block: explanation | chapter: Common Multiples
\section*{Common Multiples}
Find the common multiples of 6 and 8.

To find the common multiples of two or more number, first we write some multiples of these numbers, then we will encircle the common multiples.

Now, we write the multiples of numbers, then encircle the common multiples.

Multiples of 6 = 6, 12, 18, (24), 30, 36, 42, (48), 54, 60\\
Multiples of 8 = 8, 16, (24), 32, 40, (48), 56, 64, 72, 80

First two common multiples of 6 and 8 are 24 and 48.

\begin{displayquote}
A number that i

RANK 2
block: question | chapter: Review Exercise
10 000 + 6 000 + 700 + 80 + 5 \_\_\_\_\_\_\_\_\_\_\_\_\_\_

  \item Compare the following numbers by using symbols (<, >, =):

\end{enumerate}

\begin{enumerate}
  \setcounter{enumi}{7}
  \item Write the following numbers in an ascending order.\\

RANK 3
block

In [62]:
#CELL 36-5 Query testing
out = retrieve_math(
    "Convert 3 metres into centimetres"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])


Original Query : Convert 3 metres into centimetres
Rewritten Query: convert 3 metres into centimetres
Intent: EXPLAIN
Chapter hint: Conversion of Metres to Cenitmetres | conf: 0.4

RANK 1
block: explanation | chapter: Conversion of Metres to Cenitmetres
\section*{Conversion of Metres to Cenitmetres}
\textbf{Key Fact}\\
1m = 100cm

\begin{displayquote}
The height of Wall of China is 7 m approxmately. What will be its height in centimetres?
\end{displayquote}

\begin{displayquote}
To find the height of Wall of China in centimetres, we will multiply 7 m by 100.
\end{displayquote}

Let's convert 7 m into centimetre.

$$7\text{ m} = 7 \times 100\text{ cm}$$

$$= 700\text{ cm}$$

The students write some distances in kilometre on the writing board and

RANK 2
block: explanation | chapter: Learning Outcomes
\section*{Learning Outcomes}
\textbf{After completing this unit, you will be able to:}

\begin{itemize}
  \item Use standard metric units to measure the length of different objects.
  \item

In [63]:
#CELL 37 — Gold Evaluation Dataset
gold_eval = [
    {
        "query": "What is expanded form?",
        "relevant_keywords": ["expanded form", "place value"]
    },
    {
        "query": "Find LCM of 6 and 8",
        "relevant_keywords": ["common multiples", "lcm"]
    },
    {
        "query": "What is a proper fraction?",
        "relevant_keywords": ["proper fraction", "numerator smaller"]
    },
    {
        "query": "Convert 3 meters into centimeters",
        "relevant_keywords": ["1m = 100cm", "convert", "centimetre"]
    },
    {
        "query": "Write the place value of each digit in 200.149",
        "relevant_keywords": ["place value", "decimal"]
    }
]


In [64]:
#CELL 38 — Retrieval Evaluation
import math
def is_relevant(text, keywords):
    t = text.lower()
    return any(k.lower() in t for k in keywords)


def recall_at_k(query, keywords, k=5):
    out = retrieve_math(query, k=k)["hits"]
    return int(any(is_relevant(h["text"], keywords) for h in out))


def precision_at_k(query, keywords, k=5):
    out = retrieve_math(query, k=k)["hits"]
    rel = sum(is_relevant(h["text"], keywords) for h in out)
    return rel / k


def reciprocal_rank(query, keywords):
    out = retrieve_math(query, k=10)["hits"]
    for i,h in enumerate(out,1):
        if is_relevant(h["text"], keywords):
            return 1/i
    return 0

def ndcg_at_k(query, keywords, k=5):

    hits = retrieve_math(query, k=k)["hits"]

    # relevance list
    rels = [1 if is_relevant(h["text"], keywords) else 0 for h in hits]

    # DCG
    dcg = 0.0
    for i,rel in enumerate(rels,1):
        dcg += rel / math.log2(i+1)

    # IDCG (ideal sorted rels)
    ideal = sorted(rels, reverse=True)
    idcg = 0.0
    for i,rel in enumerate(ideal,1):
        idcg += rel / math.log2(i+1)

    if idcg == 0:
        return 0.0

    return dcg / idcg



def evaluate(gold_eval):

    recalls, precisions, mrrs, ndcgs = [], [], [], []

    for g in gold_eval:
        recalls.append(recall_at_k(g["query"], g["relevant_keywords"]))
        precisions.append(precision_at_k(g["query"], g["relevant_keywords"]))
        mrrs.append(reciprocal_rank(g["query"], g["relevant_keywords"]))
        ndcgs.append(ndcg_at_k(g["query"], g["relevant_keywords"]))

    return {
        "Recall@5": round(sum(recalls)/len(recalls),3),
        "Precision@5": round(sum(precisions)/len(precisions),3),
        "MRR": round(sum(mrrs)/len(mrrs),3),
        "nDCG@5": round(sum(ndcgs)/len(ndcgs),3)
    }


In [65]:
#Cell 39 RUN EVALUATION
print(evaluate(gold_eval))


{'Recall@5': 1.0, 'Precision@5': 0.52, 'MRR': 1.0, 'nDCG@5': 0.991}


Interpretation Layer

In [66]:
#CELL 40 Interpretation Agent
def interpret_chunks_math(query, hits, max_chunks=3):
    """
    Reads top retrieved chunks and summarizes:
    - method used
    - step style
    - terminology
    WITHOUT answering.
    """

    sample_text = "\n\n".join(
        [h["text"] for h in hits[:max_chunks]]
    )

    SYSTEM = "You analyze textbook math snippets. Do not solve."

    USER = f"""
Read the textbook snippets below.

Tell me:
1) What method is used?
2) What step style is shown?
3) Important terminology.

Do NOT answer the question.

QUESTION:
{query}

TEXT:
{sample_text}
"""

    return llm_chat(SYSTEM, USER, max_new_tokens=150)


RAT MODULES & Generation & Tool Orchestration

In [67]:
#CELL 41 — RAT Module 1 Initial Chain-of-Thought Generator
def rat_generate_initial_cot(question):

    prompt = f"""
You are a grade 4-7 math tutor.

Solve step-by-step.
Do NOT give final answer.
Only show reasoning steps.

Question:
{question}

Steps:
"""

    return llm_chat("", prompt, 200)


In [68]:
#CELL 42 — RAT Module 2 Use Thoughts as Retrieval Queries
def rat_retrieve_with_thoughts(question, cot_text):

    queries = [question]

    for line in cot_text.split("\n"):
        if len(line.strip()) > 5:
            queries.append(line.strip())

    all_hits = []
    for q in queries:
        r = retrieve_math(q)
        all_hits.extend(r["hits"])

    return all_hits


In [69]:
#CELL 43 — RAT Module 3 Verify One Step
def rat_verify_step(question, step, hits):

    context = "\n".join(
        h["text"] for h in hits[:5]
    )

    prompt = f"""
You are a strict math teacher.

Question: {question}
Reasoning Step: {step}

Textbook Context:
{context}

If step is correct, return same step.
If wrong, return corrected step only.
"""

    return llm_chat("", prompt, 150)


In [70]:
#CELL 44 — RAT Module 4 Verify All Steps (RAT Loop)
def rat_reasoning_pipeline(question):

    cot = rat_generate_initial_cot(question)

    hits = rat_retrieve_with_thoughts(question, cot)

    steps = [s for s in cot.split("\n") if s.strip()]

    verified_steps = []

    for s in steps:
        fixed = rat_verify_step(question, s, hits)
        verified_steps.append(fixed)

    return verified_steps, hits


In [71]:
#CELL 45 — RAT Module 5 Convert Verified Steps to Interpretation
def rat_steps_to_interpretation(verified_steps):

    return "\n".join(
        f"- {s}" for s in verified_steps
    )


In [72]:
# CELL 46 — RAT-Based Retrieval Wrapper

def retrieve_math_rat(query, k=6):

    # 1) Generate reasoning (RAT thoughts)
    cot = rat_generate_initial_cot(query)

    # 2) Retrieve using thoughts
    rat_hits = rat_retrieve_with_thoughts(query, cot)

    # Safety fallback
    if not rat_hits:
        return retrieve_math(query, k)

    # 3) Deduplicate by chunk id
    seen = {}
    for h in rat_hits:
        seen[h["id"]] = h

    merged = list(seen.values())

    # 4) Rerank
    intent = detect_intent_math(query)
    final_hits = rerank_math(query, merged, intent, top_k=k)

    return {
        "original_query": query,
        "rewritten_query": query,
        "intent": intent,
        "chapter_hint": None,
        "chapter_conf": 0.0,
        "hits": final_hits
    }


In [76]:
#CELL 46 HF env build
import os, getpass
from huggingface_hub import login

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    os.environ["HF_TOKEN"] = getpass.getpass("Enter HuggingFace token: ")

login(token=os.environ["HF_TOKEN"])
print("✅ Hugging Face login done")

Enter HuggingFace token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Hugging Face login done


In [77]:
#Cell 47 Import the model of meta-llama-3.1-8B-Instruct
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
FALLBACK_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

use_gpu = torch.cuda.is_available()
print("GPU available:", use_gpu)

if use_gpu:
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    load_kwargs = dict(device_map="auto", quantization_config=bnb)
else:
    load_kwargs = dict(device_map="auto")

def load_model(mid):
    tok = AutoTokenizer.from_pretrained(mid, use_fast=True)
    mdl = AutoModelForCausalLM.from_pretrained(mid, **load_kwargs)
    return tok, mdl

try:
    tokenizer, model = load_model(LLM_MODEL_ID)
    ACTIVE_MODEL_ID = LLM_MODEL_ID
except:
    tokenizer, model = load_model(FALLBACK_MODEL_ID)
    ACTIVE_MODEL_ID = FALLBACK_MODEL_ID

print("✅ Loaded:", ACTIVE_MODEL_ID)


GPU available: True


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ Loaded: meta-llama/Meta-Llama-3.1-8B-Instruct


In [78]:
# CELL 47A — Groq API (Primary LLM)

from groq import Groq
import os, getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API key: ")

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

PRIMARY_GROQ_MODEL = "llama-3.3-70b-versatile"

print("✅ Groq Llama-3.3-70B ready")


Enter Groq API key: ··········
✅ Groq Llama-3.3-70B ready


In [79]:
#Cell 48 llm_chat Helper
import torch

def llm_chat(system_prompt: str, user_prompt: str, max_new_tokens: int = 1200):

    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()},
    ]

    enc = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    # ---- HARD FORCE TENSOR EXTRACTION ----
    try:
        input_ids = enc["input_ids"]
        attention_mask = enc.get("attention_mask", None)
    except Exception:
        input_ids = enc

    if not torch.is_tensor(input_ids):
        input_ids = torch.tensor(input_ids)

    if attention_mask is not None and not torch.is_tensor(attention_mask):
        attention_mask = torch.tensor(attention_mask)

    input_ids = input_ids.to(model.device)
    if attention_mask is not None:
        attention_mask = attention_mask.to(model.device)

    prompt_len = input_ids.shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05
        )

    generated = outputs[0][prompt_len:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


In [80]:
#CELL 36-5 Query testing
out = retrieve_math_rat(
    "Convert 3 metres into centimetres"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])



The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original Query : Convert 3 metres into centimetres
Rewritten Query: Convert 3 metres into centimetres
Intent: EXPLAIN
Chapter hint: None | conf: 0.0

RANK 1
block: explanation | chapter: Conversion of Metres to Cenitmetres
\section*{Conversion of Metres to Cenitmetres}
\textbf{Key Fact}\\
1m = 100cm

\begin{displayquote}
The height of Wall of China is 7 m approxmately. What will be its height in centimetres?
\end{displayquote}

\begin{displayquote}
To find the height of Wall of China in centimetres, we will multiply 7 m by 100.
\end{displayquote}

Let's convert 7 m into centimetre.

$$7\text{ m} = 7 \times 100\text{ cm}$$

$$= 700\text{ cm}$$

The students write some distances in kilometre on the writing board and

RANK 2
block: explanation | chapter: Learning Outcomes
\section*{Learning Outcomes}
\textbf{After completing this unit, you will be able to:}

\begin{itemize}
  \item Use standard metric units to measure the length of different objects.
  \item Convert larger into smaller me

In [81]:
#CELL 36-3 Query testing

out = retrieve_math_rat(
    "What is a proper fraction?"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original Query : What is a proper fraction?
Rewritten Query: What is a proper fraction?
Intent: DEFINITION
Chapter hint: None | conf: 0.0

RANK 1
block: explanation | chapter: Proper fractions
\section*{Proper fractions}
A boy says: "I have a chocolate. I ate its 3 pieces out of 10 equal parts. It means that I have eaten $\frac{3}{10}$ of the chocolate."

An image shows a chocolate bar with some pieces broken off.

It can be shown with the help of a diagram.

A rectangle divided into 10 equal vertical strips. The last 3 strips on the right are shaded teal, representing $\frac{3}{10}$.

\begin{displayquote}
\textbf{Key Fact}

Fraction with numerator smaller than its denominator is call

RANK 2
block: rule | chapter: Mixed numbers
\subsection*{Key Fact}
A mixed number / mixed fraction consists of a whole number and a proper fraction.



RANK 3
block: rule | chapter: Improper fractions
\subsection*{Key Fact}
The fraction with numerator greater than its denominator is called improper fract

In [82]:
#CELL 36-2 Query testing

out = retrieve_math_rat(
     "What is expanded form?"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original Query : What is expanded form?
Rewritten Query: What is expanded form?
Intent: DEFINITION
Chapter hint: None | conf: 0.0

RANK 1
block: explanation | chapter: Glossary
\section*{Glossary}
\textbf{Expanded form:} To write the number as sum of place values is called expanded form.\\
\textbf{Ascending order:} The arrangement of numbers from the smallest to the greatest is called an ascending order.\\
\textbf{Descending order:} The arrangement of number from the greatest to the smallest is called descending order.\\
\textbf{Number pattern:} The number pattern is the sequence of numbers that can increase, decrease and repeat according to some rules.\\
\textbf{Prime

RANK 2
block: rule | chapter: Addition of Decimals
\subsection*{Key Fact}
To add the decimals, always write the value at the same place in a column. Add ones in ones, tenths in tenths and hundredths in hundredths.



RANK 3
block: rule | chapter: Subtraction of Decimals
\subsection*{Key Fact}
To subtract the decimals, a

In [83]:
#CELL 36-4 Query testing
out = retrieve_math_rat(
    "Find LCM of 6 and 8"
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original Query : Find LCM of 6 and 8
Rewritten Query: Find LCM of 6 and 8
Intent: PROBLEM
Chapter hint: None | conf: 0.0

RANK 1
block: explanation | chapter: Common Multiples
\section*{Common Multiples}
Find the common multiples of 6 and 8.

To find the common multiples of two or more number, first we write some multiples of these numbers, then we will encircle the common multiples.

Now, we write the multiples of numbers, then encircle the common multiples.

Multiples of 6 = 6, 12, 18, (24), 30, 36, 42, (48), 54, 60\\
Multiples of 8 = 8, 16, (24), 32, 40, (48), 56, 64, 72, 80

First two common multiples of 6 and 8 are 24 and 48.

\begin{displayquote}
A number that i

RANK 2
block: explanation | chapter: Division
\section*{Division}
84 students from a school went to visit the river side. They were given a boat to visit. 6 students could visit the river side in one round. In how many rounds will all the students visit the river?

Dividing the total number of students by 6, find out the

In [84]:
#CELL 36-1 Query testing
out = retrieve_math_rat(
    "Write the place value of each digit in 200.149 and express in expanded form."
)

print("Original Query :", out["original_query"])
print("Rewritten Query:", out["rewritten_query"])
print("Intent:", out["intent"])
print("Chapter hint:", out["chapter_hint"], "| conf:", out["chapter_conf"])

for i,h in enumerate(out["hits"],1):
    print("\nRANK",i)
    print("block:",h["meta"].get("block_type"),
          "| chapter:",h["meta"].get("chapter"))
    print(h["text"][:500])



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original Query : Write the place value of each digit in 200.149 and express in expanded form.
Rewritten Query: Write the place value of each digit in 200.149 and express in expanded form.
Intent: EXPLAIN
Chapter hint: None | conf: 0.0

RANK 1
block: practice | chapter: Numbers up to One Hundred Thousand
\subsection*{Try Yourself}
Tell the place and place value of every digit in 42 918. Also write this number in words.

'6' is at the ten thousands place and its place value $= 6 \times 10\ 000 = 60\ 000$\\
'9' is at the thousands place and its place value $= 9 \times 1\ 000 = 9\ 000$\\
'2' is at the hundreds place and its place value $= 2 \times 100 = 200$\\
'7' is at the tens place and its place value $= 7 \times 10 = 70$\\
'3' is at the ones place and its place value $= 3 \times 1 = 3$

\begin{

RANK 2
block: practice | chapter: Multiplication
\subsection*{Try Yourself}
Multiply the greatest 4-digit number with the greatest 3-digit number. Multiply the smallest 3-digit number with the 

In [85]:
import math

def is_relevant(text, keywords):
    t = text.lower()
    return any(k.lower() in t for k in keywords)

def recall_at_k(query, keywords, k=5):
    out = retrieve_math_rat(query, k=k)["hits"]
    return int(any(is_relevant(h["text"], keywords) for h in out))

def precision_at_k(query, keywords, k=5):
    out = retrieve_math_rat(query, k=k)["hits"]
    rel = sum(is_relevant(h["text"], keywords) for h in out)
    return rel / k

def reciprocal_rank(query, keywords):
    out = retrieve_math_rat(query, k=10)["hits"]
    for i,h in enumerate(out,1):
        if is_relevant(h["text"], keywords):
            return 1/i
    return 0

def ndcg_at_k(query, keywords, k=5):

    hits = retrieve_math_rat(query, k=k)["hits"]

    rels = [1 if is_relevant(h["text"], keywords) else 0 for h in hits]

    dcg = 0.0
    for i,rel in enumerate(rels,1):
        dcg += rel / math.log2(i+1)

    ideal = sorted(rels, reverse=True)
    idcg = 0.0
    for i,rel in enumerate(ideal,1):
        idcg += rel / math.log2(i+1)

    if idcg == 0:
        return 0.0

    return dcg / idcg

def evaluate_rat(gold_eval):

    recalls, precisions, mrrs, ndcgs = [], [], [], []

    for g in gold_eval:
        recalls.append(recall_at_k(g["query"], g["relevant_keywords"]))
        precisions.append(precision_at_k(g["query"], g["relevant_keywords"]))
        mrrs.append(reciprocal_rank(g["query"], g["relevant_keywords"]))
        ndcgs.append(ndcg_at_k(g["query"], g["relevant_keywords"]))

    return {
        "Recall@5": round(sum(recalls)/len(recalls),3),
        "Precision@5": round(sum(precisions)/len(precisions),3),
        "MRR": round(sum(mrrs)/len(mrrs),3),
        "nDCG@5": round(sum(ndcgs)/len(ndcgs),3)
    }


In [86]:
#Cell 39 RUN EVALUATION
print(evaluate_rat(gold_eval))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

{'Recall@5': 1.0, 'Precision@5': 0.52, 'MRR': 1.0, 'nDCG@5': 0.934}


In [87]:
#CELL 49A — Tavily Web Fallback Tool
!pip -q install tavily-python


In [88]:
#CELL 49B — Tavily Web Fallback Tool

from tavily import TavilyClient
import os, getpass

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter Tavily API key: ")

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def web_fallback_search(query):
    res = tavily.search(query, max_results=3)
    return "\n".join([r["content"] for r in res["results"]])


Enter Tavily API key: ··········


In [89]:
#CELL 50 — Context Sanitizer (same as science)
import re

BAD_LINE_PATTERNS = [
    r"please search.*",
    r"share your findings.*",
    r"collect.*",
    r"investigate.*",
    r"activity.*",
]

def sanitize_context_text(text):
    lines = (text or "").splitlines()
    keep = []

    for ln in lines:
        s = ln.strip()
        if not s:
            continue

        low = s.lower()
        if any(re.match(p, low) for p in BAD_LINE_PATTERNS):
            continue

        keep.append(s)

    out = "\n".join(keep)
    out = re.sub(r"\n{3,}", "\n\n", out)
    return out.strip()

print("✅ Math context sanitizer ready")


✅ Math context sanitizer ready


In [90]:
#CELL 51 — Select Best Chunks (Math)

def select_context_chunks_math(hits, max_chunks=3):

    chosen = []

    for h in hits:
        bt = (h.get("meta") or {}).get("block_type","").lower()

        # Priority: teaching content first
        if bt in ["explanation","rule","example","worked_example","definition"]:
            chosen.append(h)

        if len(chosen) >= max_chunks:
            break

    # Fallback if nothing matched
    if not chosen:
        chosen = hits[:max_chunks]

    # Sanitize text
    for h in chosen:
        h["text_sanitized"] = sanitize_context_text(
            h.get("text","")
        )

    return chosen


In [91]:
#CELL 52 — Python Math Tool (STEP B7)
def python_math_tool(expr):
    try:
        safe = re.sub(r"[^0-9\.\+\-\*/\(\) ]","",expr)
        val = eval(safe)

        # remove long floating tails
        if isinstance(val, float):
            val = round(val, 6)

        return str(val)
    except:
        return None



In [92]:
# CELL 53 — Math System Prompt (Autism-Friendly + Grade-Aware)

MATH_SYSTEM_PROMPT = """
You are a careful and strict elementary mathematics tutor.

STUDENT PROFILE:
- Student may struggle with understanding.
- Use very simple words.
- Short sentences.
- One idea per line.
- Step-by-step always.

STRICT RULES:
1. Use ONLY the given CONTEXT.
2. Never use outside knowledge.
3. Never guess.
4. If something is not in CONTEXT, say:
   "This is not explained in the book text I have."
5. Always show place value as:
   digit × value
   Example: 3 × 100 = 300
6. For decimals:
   - First digit after decimal = tenths
   - Second digit = hundredths
   - Third digit = thousandths
7. Finish every solution completely.
8. No half answers.
9. No long paragraphs.

FOR CALCULATION QUESTIONS:
- Show working.
- Then show final answer on its own line:
  Final Answer: ___

FOR DEFINITIONS:
- Give one sentence definition.
- Then one small example if present in context.

FOR PRACTICE REQUESTS:
- Only list questions.
- No answers.

OUTPUT FORMAT:

Step 1:
...

Step 2:
...

Final Answer:
...

Evidence: <chunk ids>
"""


In [93]:
# CELL 54 — Reasoning + Answer Builder (STEP B8)

def build_math_answer(query, interpretation, chosen_chunks, grade=4, intent=None):

    ctx_parts = []
    evidence = []

    for h in chosen_chunks:
        meta = h.get("meta") or {}
        cid = meta.get("chunk_id","")
        evidence.append(cid)

        ctx_parts.append(
            f"[{cid}] (chapter: {meta.get('chapter','')})\n{h['text_sanitized']}"
        )

    context = "\n\n---\n\n".join(ctx_parts)

    USER = f"""
GRADE: {grade}
QUESTION TYPE: {intent}

QUESTION:
{query}

INTERPRETATION:
{interpretation}

CONTEXT:
{context}

TASK:
Use the same method shown in the context.
Follow the same step style.
If numbers contain decimals or fractions, explain positions clearly.
Follow SYSTEM rules exactly.
Solve step-by-step.
Check each step before moving on.
Finish completely.
End with:
Evidence: {", ".join(evidence)}
"""

    answer = llm_chat(
        MATH_SYSTEM_PROMPT,
        USER,
        max_new_tokens=1200
    )

    return answer


In [94]:
# CELL 55 — Answer Formatter

def format_answer(text):
    if not text:
        return text

    text = text.replace("**","")
    text = text.replace("Step", "\nStep")
    text = text.replace("\n\n\n","\n\n")

    return text.strip()


In [95]:
#CELL 56 — Critic / Validator (STEP B9)
def critic_math(question, answer):

    SYSTEM = """
You are a strict math teacher.
Check:
- calculations
- place values
- units
- fraction steps
- decimal handling

If any step is wrong, reply NO.
Otherwise YES.
"""
    USER = f"""
Question: {question}
Answer: {answer}

Is this correct and consistent with textbook context?
Reply only YES or NO.
"""

    out = llm_chat(SYSTEM, USER, 60)
    return "YES" in out.upper()

print("✅ Math critic ready")


✅ Math critic ready


In [96]:
#CELL 57 — Personalization Layer (STEP B10)
def personalize_answer(text):
    SYSTEM = "Rewrite in simple Grade-4 language."
    return llm_chat(SYSTEM, text, 600)


In [97]:
# CELL 58 — Full Generation Pipeline (STEP B7 → B11)
import re

def math_generate(query):

    # ---------------------------------------------------
    # STEP B2–B5 → Base Retrieval (for router + metadata)
    # ---------------------------------------------------
    r = retrieve_math(query)
    intent = r["intent"]

    # ---------------------------------------------------
    # 🧠 RAT STEP 1–4 → Generate + Verify Reasoning
    # ---------------------------------------------------
    verified_steps, rat_hits = rat_reasoning_pipeline(query)

    # Convert verified steps into interpretation text
    interpretation = rat_steps_to_interpretation(verified_steps)

    # Select context using RAT retrieval results
    chosen = select_context_chunks_math(rat_hits)

    # ---------------------------------------------------
    # STEP B7 → PRACTICE SHORT-CIRCUIT
    # ---------------------------------------------------
    if intent == "PRACTICE":
        raw = llm_chat(
            MATH_SYSTEM_PROMPT,
            f"Create 5 practice questions about: {query}. No answers.",
            200
        )
        return format_answer(raw)

    # ---------------------------------------------------
    # STEP B7 → Python Math Tool (auto trigger)
    # ---------------------------------------------------
    py_result = None
    if re.search(r"\d", query) and any(op in query for op in ["÷","/","×","*","+","-"]):
        expr = query.replace("÷","/").replace("×","*")
        py_result = python_math_tool(expr)

    # ---------------------------------------------------
    # STEP B8 → Build Answer using VERIFIED STEPS
    # ---------------------------------------------------
    answer = build_math_answer(
        query,
        interpretation,
        chosen,
        grade=4,
        intent=intent
    )

    if py_result:
        answer += f"\n\nFinal Answer: {py_result}"

    # ---------------------------------------------------
    # STEP B9 → Critic (single retry)
    # ---------------------------------------------------
    if not critic_math(query, answer):

        answer = build_math_answer(
            query,
            interpretation,
            chosen,
            grade=4,
            intent=intent
        )

        if py_result:
            answer += f"\n\nFinal Answer: {py_result}"

    # ---------------------------------------------------
    # STEP B10 → Personalization
    # ---------------------------------------------------
    answer = personalize_answer(answer)

    # ---------------------------------------------------
    # STEP B10.5 → Formatting
    # ---------------------------------------------------
    answer = format_answer(answer)

    # ---------------------------------------------------
    # WEB FALLBACK (only if book failed)
    # ---------------------------------------------------
    if "not explained in the book text" in answer.lower():
        web_info = web_fallback_search(query)
        answer += "\n\nExtra help (from web):\n" + web_info

    # ---------------------------------------------------
    # STEP B11 → Return
    # ---------------------------------------------------
    return answer


In [98]:
print(math_generate("Find 12 × 8"))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Let's Learn Multiplication.

Step 1: We want to find the answer to 12 multiplied by 8.

Step 2: To make it easier, let's break down 12 into two parts: 10 and 2. And 8 is just 8.

Step 3: First, let's multiply 10 by 8. That's like having 10 groups of 8 toys each. We get 80 toys!

Step 4: Now, let's multiply 2 by 8. That's like having 2 groups of 8 toys each. We get 16 toys!

Step 5: Finally, let's add the toys together. We have 80 toys from the first group and 16 toys from the second group. So, we add them together: 80 + 16 = 96 toys!

Answer: 12 multiplied by 8 is equal to 96.


In [99]:
# CELL 59A — Color Printer

def c(text, color):
    colors = {
        "cyan": "\033[96m",
        "green": "\033[92m",
        "yellow": "\033[93m",
        "red": "\033[91m",
        "blue": "\033[94m",
        "magenta": "\033[95m",
        "bold": "\033[1m",
        "end": "\033[0m"
    }
    return colors.get(color,"") + text + colors["end"]


In [100]:
# CELL 59B — Debug Wrapper Around math_generate (RAT VERSION)

def math_generate_debug(query):

    print(c("="*90,"cyan"))
    print(c("ORIGINAL QUERY: ","bold")+query)

    # ---------- Router ----------
    r = retrieve_math(query)

    print(c("\n[Router]","blue"))
    print("Intent:", c(r["intent"],"yellow"))
    print("Chapter hint:", r["chapter_hint"], "| confidence:", r["chapter_conf"])

    # ---------- RAT Initial CoT ----------
    cot = rat_generate_initial_cot(query)
    print(c("\n[RAT Initial Reasoning]","magenta"))
    print(cot)

    # ---------- RAT Retrieval ----------
    rat_hits = rat_retrieve_with_thoughts(query, cot)

    print(c("\n[RAT Retrieved Chunks]","blue"))
    for i,h in enumerate(rat_hits[:3],1):
        print(c(f"\nRANK {i}","magenta"))
        print("block:", h["meta"].get("block_type"),
              "| chapter:", h["meta"].get("chapter"))
        print(h["text"][:300])

    # ---------- RAT Step Verification ----------
    steps = [s for s in cot.split("\n") if s.strip()]
    verified_steps = []

    print(c("\n[RAT Step Verification]","blue"))

    for s in steps:
        fixed = rat_verify_step(query, s, rat_hits)
        print(c("Original: ","yellow")+s)
        print(c("Verified: ","green")+fixed)
        print("-"*60)
        verified_steps.append(fixed)

    interpretation = rat_steps_to_interpretation(verified_steps)

    # ---------- Python Tool ----------
    py_result = None
    if re.search(r"\d", query) and any(op in query for op in ["÷","/","×","*","+","-"]):
        expr = query.replace("÷","/").replace("×","*")
        py_result = python_math_tool(expr)

    if py_result:
        print(c("\n[Python Tool Used]","green"))
        print("Expression:", expr)
        print("Result:", py_result)
    else:
        print(c("\n[Python Tool] Not used","yellow"))

    # ---------- Build Answer ----------
    chosen = select_context_chunks_math(rat_hits)

    answer = build_math_answer(
        query,
        interpretation,
        chosen,
        grade=4,
        intent=r["intent"]
    )

    if py_result:
        answer += f"\n\nFinal Answer: {py_result}"

    # ---------- Critic ----------
    ok = critic_math(query, answer)
    print(c("\n[Critic Check]: ","blue")+ (c("PASS","green") if ok else c("FAIL","red")))

    # ---------- Personalize ----------
    answer = personalize_answer(answer)
    answer = format_answer(answer)

    # ---------- Web Fallback ----------
    if "not explained in the book text" in answer.lower():
        print(c("\n[WEB FALLBACK USED]","red"))
        web_info = web_fallback_search(query)
        answer += "\n\nExtra help (from web):\n" + web_info
    else:
        print(c("\n[WEB FALLBACK] Not used","green"))

    # ---------- Final ----------
    print(c("\n[FINAL ANSWER]","green"))
    print(answer)

    print(c("="*90,"cyan"))


In [101]:
#Cell 60-1 Testing
math_generate_debug(
    "Write the place value of each digit in 200.149"
)


ORIGINAL QUERY: Write the place value of each digit in 200.149


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Router]
Intent: EXPLAIN
Chapter hint: Numbers up to One Hundred Thousand | confidence: 0.3

[RAT Initial Reasoning]
To find the place value of each digit, we need to understand the concept of place value. 

The given number is 200.149. We can break it down into its individual digits and their corresponding place values:

- The digit 2 is in the hundreds place.
- The digit 0 is in the tens place.
- The digit 0 is in the ones place.
- The decimal point separates the whole numbers from the decimal part.
- The digit 1 is in the tenths place.
- The digit 4 is in the hundredths place.
- The digit 9 is in the thousandths place.

So, the place value of each digit is as follows:

- 2 (hundreds place)
- 0 (tens place)
- 0 (ones place)
- 1 (tenths place)
- 4 (hundredths place)
- 9 (thousandths place)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: practice | chapter: Numbers up to One Hundred Thousand
\subsection*{Try Yourself}
Tell the place and place value of every digit in 42 918. Also write this number in words.

'6' is at the ten thousands place and its place value $= 6 \times 10\ 000 = 60\ 000$\\
'9' is at the thousands place and its place value $= 9 \times 1\ 000 = 9\ 000$\\
'2' is at the 

RANK 2
block: explanation | chapter: Numbers up to One Hundred Thousand
\section*{Numbers up to One Hundred Thousand}
The image shows a boy asking a question:

\begin{displayquote}
The length of the great wall of China is 21 196 km. How can we read and write 21 196 in words?
\end{displayquote}

The image shows a girl explaining:

\begin{displayquote}
The number which is

RANK 3
block: practice | chapter: Comparing and Ordering Numbers
\subsection*{Try Yourself}
Compare 8 799; 22 234 and 22 229.

The price of three mobile phone sets are Rs 62 870, Rs 78 200 and Rs 75 110 respectively. Compare their

...

Original: - 4 (hundredths place)
Verified: Given the number 200.149, let's find the place value of each digit.

The given number is 200.149. 

- The digit '2' is at the hundred thousands place and its place value = 2 × 100 000 = 2 000 000
- The digit '0' is at the ten thousands place and its place value = 0 × 10 000 = 0
- The digit '0' is at the thousands place and its place value = 0 × 1 000 = 0
- The digit '1' is at the hundreds place and its place value = 1 × 100 = 100
- The digit '4' is at the tens
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: - 9 (thousandths place)
Verified: Given the number 200.149, let's find the place value of each digit.

The given number is 200.149. 

- The digit '2' is at the hundredths place and its place value $= 2 \times 100 = 200$
- The digit '0' is at the tenths place and its place value $= 0 \times 10 = 0$
- The digit '1' is at the ones place and its place value $= 1 \times 1 = 1$
- The digit '4' is at the thousandths place and its place value $= 4 \times 0.001 = 0.004$
- The digit '9' is
------------------------------------------------------------

[Python Tool] Not used


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: FAIL

[WEB FALLBACK] Not used

[FINAL ANSWER]
Let's break down the number 200.149 into its individual parts.

Step 1: The number 200.149 has many parts. Let's find out what each part is worth.

Step 2: The first part is 2. It's like a big group of 100,000. So, 2 x 100,000 = 200,000.

Step 3: The next part is 0. It's like a group of 10,000. But since it's 0, it doesn't add anything.

Step 4: The next part is also 0. It's like a group of 1,000. Again, since it's 0, it doesn't add anything.

Step 5: Now we have 1. It's like a group of 100. So, 1 x 100 = 100.

Step 6: Next is 4. It's like a group of 10. So, 4 x 10 = 40.

Step 7: Then we have 9. It's like a single item. So, 9 x 1 = 9.

Step 8: After that is 9. But this time it's like a tenth of an item. So, 9 x 0.1 = 0.9.

Step 9: Next is 4. This time it's like a hundredth of an item. So, 4 x 0.01 = 0.04.

Step 10: Finally, we have 1. This time it's like a thousandth of an item. So, 1 x 0.001 = 0.001.

Now we know what each

In [102]:
#Cell 60-2 Testing
math_generate_debug(
    "What is expanded form?"
)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ORIGINAL QUERY: What is expanded form?

[Router]
Intent: DEFINITION
Chapter hint: Place Value of Digits in Decimals | confidence: 0.2

[RAT Initial Reasoning]
Let's break it down step by step.

Expanded form is a way to write a number using its place value. It shows the value of each digit in relation to its position.

For example, let's take the number 456.

In standard form, it's written as 456.

To write it in expanded form, we need to multiply each digit by its corresponding place value (hundreds, tens, ones).

So, we can break it down like this:

- The digit 4 is in the hundreds place, so its value is 4 x 100 =?
- The digit 5 is in the tens place, so its value is 5 x 10 =?
- The digit 6 is in the ones place, so its value is 6 x 1 =?

Now, let's calculate the values:

- 4 x 100 = 400
- 5 x 10 = 50
- 6 x 1 = 6

Now that


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: explanation | chapter: Glossary
\section*{Glossary}
\textbf{Expanded form:} To write the number as sum of place values is called expanded form.\\
\textbf{Ascending order:} The arrangement of numbers from the smallest to the greatest is called an ascending order.\\
\textbf{Descending order:} The arrangement of number from the great

RANK 2
block: explanation | chapter: Improper fractions
\section*{Improper fractions}
A girl says: "Consider the following figures:"

Two diagrams are shown:

\begin{enumerate}
  \item A square divided into three equal vertical rectangles. All three rectangles are shaded pink.
$$\frac{3}{3}$$

  \item Two squares divided into four equal triangles by thei

RANK 3
block: explanation | chapter: Data Handling
\section*{Data Handling}



[RAT Step Verification]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Original: - 6 x 1 = 6
Verified: The given reasoning step is: 
- 6 x 1 = 6

This step is correct. The expanded form of the number 6 is indeed 6 (ones place).
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: Now that
Verified: Now that we have discussed various concepts in the textbook, let's focus on the question at hand: What is expanded form?

According to the glossary provided in the textbook, expanded form refers to writing a number as a sum of its place values. This means that instead of representing a number in its standard form (e.g., 456), we break it down into its individual place values (e.g., 400 + 50 + 6).

Therefore, the answer to the question is:

Expanded form: Writing a number as a sum of its place values.
------------------------------------------------------------

[Python Tool] Not used


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: PASS

[WEB FALLBACK] Not used

[FINAL ANSWER]
Let's learn about expanded form.

What is Expanded Form?

Expanded form is a way to write a number by adding up all its parts. Each part is like a little house where numbers live, and we call these houses "place values".

How Do We Write Numbers in Expanded Form?

Let's take the number 456. We can break it down into three parts:

*   400 (this is like a big house with 4 numbers living inside)
*   50 (this is like a medium-sized house with 5 numbers living inside)
*   6 (this is like a small house with just 1 number living inside)

When we add these three houses together, we get the number 456!

So, we can write 456 in expanded form as: 400 + 50 + 6.

Why Is Expanded Form Important?

Expanded form helps us understand how each number in a number contributes to the overall value of the number. It's like being a detective and figuring out the mystery of how all the numbers fit together!


In [103]:
#Cell 60-3 Testing
math_generate_debug(
    "Find LCM of 6 and 8"
)


ORIGINAL QUERY: Find LCM of 6 and 8

[Router]
Intent: PROBLEM
Chapter hint: Common Multiples | confidence: 0.2


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Initial Reasoning]
To find the Least Common Multiple (LCM) of 6 and 8, we need to follow these steps:

1. List the multiples of each number:
   Multiples of 6: 6, 12, 18, 24, 30,...
   Multiples of 8: 8, 16, 24, 32,...

2. Identify the smallest multiple that is common to both lists:
   The first number that appears in both lists is 24.

3. Since 24 is the smallest multiple that is common to both lists, it is also the LCM of 6 and 8.

However, we can also use the prime factorization method to find the LCM:

1. Find the prime factors of each number:
   Prime factors of 6: 2 * 3
   Prime factors of 8: 2^3

2. Multiply the highest power of each prime factor that appears in either list:


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: explanation | chapter: Common Multiples
\section*{Common Multiples}
Find the common multiples of 6 and 8.

To find the common multiples of two or more number, first we write some multiples of these numbers, then we will encircle the common multiples.

Now, we write the multiples of numbers, then encircle the common multiples.

Multiples o

RANK 2
block: question | chapter: Review Exercise
10 000 + 6 000 + 700 + 80 + 5 \_\_\_\_\_\_\_\_\_\_\_\_\_\_

  \item Compare the following numbers by using symbols (<, >, =):

\end{enumerate}

\begin{enumerate}
  \setcounter{enumi}{7}
  \item Write the following numbers in an ascending order.\\

RANK 3
block: explanation | chapter: Pie Chart
\subsection*{Pie Chart}
The following table represents the data shown in the pie chart:

\begin{displayquote}
\textbf{Key Fact}\\
A pie chart is also called a circle graph. It can be divided into many sectors and each sector represents only one type of thing.
\end{displayqu

...

Original:    Prime factors of 8: 2^3
Verified: I see you're trying to find the LCM of 6 and 8. Let's get back on track.

To find the LCM, we need to find the prime factors of both numbers. 

Prime factors of 6: 2 * 3
Prime factors of 8: 2^3

Now, let's list the multiples of 6 and 8 to find the common multiples.

Multiples of 6 = 6, 12, 18, 24, 30, 36, 42, 48, 54, 60
Multiples of 8 = 8, 16, 24, 32, 40, 48, 56, 64,
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: 2. Multiply the highest power of each prime factor that appears in either list:
Verified: I see you're trying to find the LCM of 6 and 8. Let's break it down step by step.

First, we need to find the prime factors of both numbers:

6 = 2 × 3
8 = 2²

Now, we need to find the highest power of each prime factor that appears in either list:

Highest power of 2: 2² (from 8)
Highest power of 3: 3 (from 6)

Now, we multiply these highest powers together:

LCM = 2² × 3
= 4 × 3
= 12

Therefore, the LCM of 6 and 8 is 12.

Now, let's move on
------------------------------------------------------------

[Python Tool] Not used


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: FAIL

[WEB FALLBACK] Not used

[FINAL ANSWER]
Let's find the least common multiple (LCM) of 6 and 8.

Step 1: Write the multiples of 6 and 8.

Multiples of 6: 6, 12, 18, 24, 30, 36, 42, 48, 54, 60
Multiples of 8: 8, 16, 24, 32, 40, 48, 56, 64, 72, 80

Step 2: Find the numbers that are in both lists.

Multiples of 6: 6, 12, 18, 24, 30, 36, 42, 48, 54, 60
Multiples of 8: 8, 16, 24, 32, 40, 48, 56, 64, 72, 80

Step 3: Identify the first two numbers that are in both lists.

The first two numbers that are in both lists are 24 and 48.

Step 4: The least common multiple (LCM) is the smallest number that is in both lists.

Since 24 is smaller than 48, the LCM of 6 and 8 is 24.

So, the final answer is: 24


In [104]:
#Cell 60-4 Testing
math_generate_debug(
    "Convert 3 meters into centimeters"
)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ORIGINAL QUERY: Convert 3 meters into centimeters

[Router]
Intent: EXPLAIN
Chapter hint: Conversion of Metres to Cenitmetres | confidence: 0.3

[RAT Initial Reasoning]
To convert 3 meters into centimeters, we need to know the conversion factor between meters and centimeters. 

1 meter is equal to 100 centimeters.

So, we can multiply 3 meters by 100 centimeters to get the equivalent in centimeters.

First, let's multiply 3 by 100:

3 x 100 =?

We can break this down further by multiplying 3 by 100 as (3 x 100)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: explanation | chapter: Conversion of Metres to Cenitmetres
\section*{Conversion of Metres to Cenitmetres}
\textbf{Key Fact}\\
1m = 100cm

\begin{displayquote}
The height of Wall of China is 7 m approxmately. What will be its height in centimetres?
\end{displayquote}

\begin{displayquote}
To find the height of Wall of China in centimetres, we will multiply 7

RANK 2
block: rule | chapter: Conversions of Weeks into Days
\subsection*{Key Fact}
\begin{itemize}
  \item 1 day = 24 hours
  \item 1 week = 7 days
  \item 1 month = 30 days
  \item 1 year = 12 months
\end{itemize}

\textbf{Convert 8 weeks 20 days into days.}

\begin{displayquote}
To convert 8 weeks 20 days into days, first we convert 8 weeks into days and t

RANK 3
block: explanation | chapter: Division of Decimals by Whole Numbers
\section*{Division of Decimals by Whole Numbers}
\begin{displayquote}
Komal divides 6.6 kg apples in 3 baskets. How many kilograms of apples are there in each bas

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Original: 3 x 100 =?
Verified: Since you have provided the reasoning step as "3 x 100 =", I will verify if it is correct.

The given key fact is that 1 meter is equal to 100 centimeters. To convert 3 meters into centimeters, we need to multiply 3 by 100.

So, the correct reasoning step is:

3 x 100 = 300

Therefore, the height of the wall in centimeters is 300 cm.
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: We can break this down further by multiplying 3 by 100 as (3 x 100)
Verified: Since you have provided the reasoning step for converting 3 meters into centimeters, I will verify it.

The given step is: 
We can break this down further by multiplying 3 by 100 as (3 x 100)

This step is correct. The conversion factor is indeed 1 meter = 100 centimeters, so multiplying 3 meters by 100 will give us the equivalent length in centimeters.

Therefore, the correct step remains the same:

We can break this down further by multiplying 3 by 100 as (3 x 100)
------------------------------------------------------------

[Python Tool] Not used


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: PASS

[WEB FALLBACK] Not used

[FINAL ANSWER]
Let's convert 3 meters into centimeters.

Did you know that 1 meter is equal to 100 centimeters? That's like a big step of 100 little steps!

Now, let's take 3 meters and multiply it by 100 centimeters. We can write it as 3 × 100.

When we multiply 3 by 100, we get 300. So, 3 meters is equal to 300 centimeters.

That's our answer!


In [105]:
#Cell 60-5 Testing
math_generate_debug(
    "Solve 456 ÷ 6"
)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ORIGINAL QUERY: Solve 456 ÷ 6

[Router]
Intent: PROBLEM
Chapter hint: Conversion of Hours into Minutes | confidence: 0.1

[RAT Initial Reasoning]
To solve the division problem 456 ÷ 6, we can use the following steps:

1. We need to find out how many groups of 6 we can make from 456.
2. Since 6 is a smaller number than 456, we will divide 456 by 6.
3. To do this, we can start by dividing the hundreds place (400) by 6.
   400 ÷ 6 =?
   We can think of 400 as 6 groups of 66 (since 6 x 66 = 396).
4. Now, let's look at the tens place (50).
   50 ÷ 6 =?
   We can think of 50 as 6 groups of 8 (since 6 x 8 = 48).
5. Finally, let's look at the ones place (6).
   6 ÷ 6 = 1 group of 6.
6. Now, let's put


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: explanation | chapter: Division
\section*{Division}
84 students from a school went to visit the river side. They were given a boat to visit. 6 students could visit the river side in one round. In how many rounds will all the students visit the river?

Dividing the total number of students by 6, find out the number of rounds taken 

RANK 2
block: practice | chapter: Numbers up to One Hundred Thousand
\subsection*{Try Yourself}
Tell the place and place value of every digit in 42 918. Also write this number in words.

'6' is at the ten thousands place and its place value $= 6 \times 10\ 000 = 60\ 000$\\
'9' is at the thousands place and its place value $= 9 \times 1\ 000 = 9\ 000$\\
'2' is at the 

RANK 3
block: explanation | chapter: Learning Outcomes
\section*{Learning Outcomes}
\textbf{After completing this section, you will be able to:}

\begin{itemize}
  \item Add numbers up to 5-digit.
  \item Solve real life number stories involving addition o

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Original:    6 ÷ 6 = 1 group of 6.
Verified: I see you're trying to solve some division problems. Let's go through them step by step.

1. 456 ÷ 6

Textbook Context:
84 students from a school went to visit the river side. They were given a boat to visit. 6 students could visit the river side in one round. In how many rounds will all the students visit the river?

Dividing the total number of students by 6, find out the number of rounds taken by the boat, so that all the students will have a boat ride.

Number of students visited the river side = 84\\
Number of students who could visit the river side in one round = 6\\
Total number of rounds = 84 ÷ 6
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: 6. Now, let's put
Verified: Since the question is to solve 456 ÷ 6, let's follow the steps provided in the textbook context.

6. Now, let's put

Textbook Context:
\section*{Division}
84 students from a school went to visit the river side. They were given a boat to visit. 6 students could visit the river side in one round. In how many rounds will all the students visit the river?

Dividing the total number of students by 6, find out the number of rounds taken by the boat, so that all the students will have a boat ride.

Number of students visited the river side = 84\\
Number of students who could visit the river side in one round = 6\\
Total number
------------------------------------------------------------

[Python Tool Used]
Expression: Solve 456 / 6
Result: 76.0


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: PASS

[WEB FALLBACK] Not used

[FINAL ANSWER]
Let's do a fun math problem together. We are going to divide 456 by 6.

Step 1: 
We start with the biggest number, which is 6. When we divide 6 by 6, we get 1.

Step 2: 
We know that 1 times 6 is 6. So, we write 1 as our answer and put 6 below it.

Step 3: 
Now, we subtract 6 from 6. That means we have nothing left, so it's 0.

Step 4: 
Next, we look at the next big number, which is 50. When we divide 50 by 6, we get 8.

Step 5: 
We know that 8 times 6 is 48. So, we write 8 in our answer and put 48 below it. Then, we subtract 48 from 50. That leaves us with 2.

Step 6: 
Now, we look at the next big number, which is 20. When we divide 20 by 6, we get 3.

Step 7: 
We know that 3 times 6 is 18. So, we write 3 in our answer and put 18 below it. Then, we subtract 18 from 20. That leaves us with 2.

Step 8: 
Next, we look at the next big number, which is 40. When we divide 40 by 6, we get 6.

Step 9: 
We know that 6 times 6 is 36

In [106]:
#Cell 60-6 Testing
math_generate_debug(
    "Why do we find common denominator?"
)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ORIGINAL QUERY: Why do we find common denominator?

[Router]
Intent: PROBLEM
Chapter hint: Simplification of Fractions | confidence: 0.2

[RAT Initial Reasoning]
To solve this question, let's break it down step by step:

1. **Understanding the concept of fractions**: Fractions are used to represent parts of a whole. For example, 1/2 represents one out of two equal parts.

2. **Adding or subtracting fractions**: When we add or subtract fractions, we need to make sure that they have the same denominator (the bottom number). This is because the denominators tell us how many equal parts something is divided into.

3. **Problem with different denominators**: If we try to add or subtract fractions with different denominators, we can't compare them directly. For instance, if we want to add 1/4 and 1/6, we can't just add the numerators (1 + 1) because the denominators are different.

4. **Need for a common denominator**: To add or subtract fractions, we need to find a common denominator that b

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: practice | chapter: Addition of Fractions
\subsection*{Try Yourself}
Add the following fractions:\\
(a) $\frac{3}{4} + \frac{1}{4}$\\
(b) $\frac{9}{10} + \frac{7}{10}$\\
(c) $\frac{3}{9} + \frac{7}{9}$

\noindent\rule{\textwidth}{0.5pt}



RANK 2
block: practice | chapter: Conversion of a decimal into a fraction
\subsection*{Try Yourself}
\begin{displayquote}
The highest temperature of Multan is 44.1°C. what will be this temperature in fraction?
\end{displayquote}



RANK 3
block: question | chapter: Review Exercise
There are 20 family members in total while people preffering chocolate ice-cream are 4. How will you show this number as a fraction?



Mathematics 4 Answers

[RAT Step Verification]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Original: 3. **Problem with different denominators**: If we try to add or subtract fractions with different denominators, we can't compare them directly. For instance, if we want to add 1/4 and 1/6, we can't just add the numerators (1 + 1) because the denominators are different.
Verified: Now, let's get back to the main topic - finding a common denominator.

**Why do we find a common denominator?**

If we try to add or subtract fractions with different denominators, we can't compare them directly. For instance, if we want to add 1/4 and 1/6, we can't just add the numerators (1 + 1) because the denominators are different.

To solve this problem, we need to find a common denominator that both fractions can use. This way, we can compare the fractions directly and perform the addition or subtraction.

For example, to add 1/4 and 1/6, we can find the least common multiple (LCM) of 4 and
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: 4. **Need for a common denominator**: To add or subtract fractions, we need to find a common denominator that both fractions can use. This way, we can compare
Verified: Now, let's get back to the question at hand: "Why do we find common denominator?"

**Answer:** We find a common denominator to add or subtract fractions because it allows us to compare the fractions directly. Without a common denominator, we cannot determine if the fractions are equal or not.

Let's take an example: $\frac{1}{2} + \frac{1}{3}$. If we don't have a common denominator, we cannot add these fractions directly. But if we find a common denominator, say 6, we can rewrite the fractions as $\frac{3}{6} + \frac{2}{6}$, and now we can add them easily.

So, finding a common denominator is essential when adding or
------------------------------------------------------------

[Python Tool] Not used


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: PASS

[WEB FALLBACK] Not used

[FINAL ANSWER]
Why Do We Need a Common Denominator?

When we want to add or subtract fractions, we need to make sure they have the same bottom number, called the denominator. If the denominators are different, we can't directly add or compare the fractions.

What Happens If We Don't Have the Same Denominator?

Imagine you have two pizzas. One pizza has 4 slices, and the other pizza has 5 slices. You can't directly compare how much pizza you have if you're using different sized slices.

How Do We Solve This Problem?

To solve this problem, we need to find a common denominator, which is the smallest number that both denominators can divide into evenly. For example, if we have $\frac{1}{4}$ and $\frac{3}{5}$, we need to find a common denominator, which is 20.

Now We Can Compare!

With a common denominator of 20, we can rewrite the fractions as $\frac{5}{20}$ and $\frac{12}{20}$. Now we can easily add them together.

Why Is Finding a Common 

In [107]:
#Cell 60-7 Testing
math_generate_debug(
    "Give me 5 practice questions on fractions"
)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


ORIGINAL QUERY: Give me 5 practice questions on fractions

[Router]
Intent: EXPLAIN
Chapter hint: Comparing unlike fractions | confidence: 0.2

[RAT Initial Reasoning]
Here are five practice questions on fractions for you to solve:

1. Emily has 12 cookies that she wants to share with her friends. She gives 1/4 of the cookies to her friend Sarah. How many cookies does Sarah get?

2. A pizza parlor has 16 slices of pizza left after serving customers. If they want to package 1/8 of the remaining slices in a special box, how many slices will be in the box?

3. A bookshelf has 24 books on it. If 1/6 of the books are fiction and 1/4 of the books are non-fiction, what fraction of the books are either fiction or non-fiction?

4. Tom has 18 pencils in his pencil case. He gives 1/3 of the pencils to his brother. What fraction of the pencils does Tom have left?

5. A recipe calls for 1/2 cup of sugar. If you need to make 3/4 of the recipe


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[RAT Retrieved Chunks]

RANK 1
block: explanation | chapter: Conversion of Hours into Minutes
\subsection*{Conversion of Minutes into Seconds}
\begin{displayquote}
Saad solved the division question in 6 min. In how many seconds did he solve the question?
\end{displayquote}

To convert minutes into seconds, we multiply minutes by 60.

$$6 \text{min} = 6 \times 60 \text{ sec}$$

$$= 360 \text{

RANK 2
block: explanation | chapter: Time
\section*{Time}
\begin{displayquote}
It's time for the science experiment. Hour hand is on 10 and minute hand is on 2. It means that the time is 10 minutes past 10. What is this third hand used for?
\end{displayquote}

The third hand of the clock is second hand. When the second hand moves from one s

RANK 3
block: explanation | chapter: Completing Symmetrical Figures:
\section*{Completing Symmetrical Figures:}
\begin{displayquote}
If one half of a symmetrical figure is given, we can complete it by drawing its remaining half.
\end{displayquote}

\textbf{Ke

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Original: 4. Tom has 18 pencils in his pencil case. He gives 1/3 of the pencils to his brother. What fraction of the pencils does Tom have left?
Verified: I see you're trying to distract me with all those unrelated topics, but I'm not falling for it. You want to practice fractions, don't you? Alright, let's get down to business. Here are five practice questions on fractions:

1. Tom has 18 pencils in his pencil case. He gives 1/3 of the pencils to his brother. What fraction of the pencils does Tom have left?

Answer: To find the fraction of pencils Tom has left, we need to subtract 1/3 from 1 (the whole). So, Tom has 1 - 1/3 = 2/3 of the pencils left.

2. A pizza has 12 slices. If 3/4 of the slices are
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Original: 5. A recipe calls for 1/2 cup of sugar. If you need to make 3/4 of the recipe
Verified: I see you're trying to distract me with all those unrelated topics, but I'm not falling for it. You want to practice fractions, don't you? Alright, let's get down to business. Here are five practice questions on fractions:

1. Simplify the fraction: $\frac{12}{16}$

2. A recipe calls for 1/2 cup of sugar. If you need to make 3/4 of the recipe, how much sugar will you need?

3. Compare the fractions: $\frac{1}{2}$ and $\frac{2}{4}$

4. A pizza has 8 slices. If 3/4 of the pizza is eaten, how many slices are left?

5. Simplify
------------------------------------------------------------

[Python Tool] Not used


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



[Critic Check]: FAIL

[WEB FALLBACK] Not used

[FINAL ANSWER]
Let's practice fractions together.

Question 1: Simplify the fraction $\frac{18}{24}$.

To simplify a fraction, we need to find the biggest number that both numbers have in common. This number is 6. Now, we can divide both numbers by 6.

$\frac{18}{24} = \frac{18 \div 6}{24 \div 6}$
$= \frac{3}{4}$

So, the simplified fraction is $\frac{3}{4}$.

Question 2: A recipe calls for $\frac{3}{8}$ cup of sugar. If you want to make $\frac{3}{4}$ of the recipe, how much sugar will you need?

To find the answer, we need to multiply the fraction of the recipe by the amount of sugar called for.

$\frac{3}{4} \times \frac{3}{8} = \frac{3 \times 3}{4 \times 8}$
$= \frac{9}{32}$

So, you will need $\frac{9}{32}$ cup of sugar.

Question 3: Compare the fractions $\frac{1}{2}$ and $\frac{2}{4}$.

To compare fractions, we need to find a common denominator. The common denominator of 2 and 4 is 4. Now, we can rewrite both fractions with a denomi